In [1]:
import pandas as pd
import numpy as np
import os

# Configuration

METRICS = {
    "IGD_Mean": True,   # True  -> minimize
    "HV_Mean": False    # False -> maximize
}

# Load result files (local paths)

RESULTS_DIR = "./results"

files = {
    "SGPSO": os.path.join(RESULTS_DIR, "SGPSO", "summary.csv"),
    "MGPSO": os.path.join(RESULTS_DIR, "MGPSO", "summary.csv"),
    "SGMGPSO": os.path.join(RESULTS_DIR, "SGMGPSO", "summary.csv"),
    "SGMGPSO_PD": os.path.join(RESULTS_DIR, "SGMGPSO_PP_PD", "summary.csv"),
    "SGMGPSO_TVSR": os.path.join(RESULTS_DIR, "SGMGPSO_TVSR", "summary.csv"),
    "SGMGPSO_FULL": os.path.join(RESULTS_DIR, "SGMGPSO_FULL", "summary.csv")
}

# Load and merge datasets

all_df = []

for model_name, path in files.items():

    temp = pd.read_csv(path)

    temp.columns = temp.columns.str.strip()

    temp["Model"] = model_name

    all_df.append(temp)

df = pd.concat(all_df, ignore_index=True)

# Process metrics

for METRIC, MINIMIZE in METRICS.items():

    print("\n" + "="*100)
    print(f"RANK COMPARISON TABLE ({METRIC})")
    print("="*100)

    # Select relevant columns

    metric_df = df[["Function", "Model", METRIC]].copy()

    # Compute ranks

    metric_df["Rank"] = metric_df.groupby("Function")[METRIC] \
        .rank(method="min", ascending=MINIMIZE)

    # Pivot to comparison table

    comparison_table = metric_df.pivot(
        index="Function",
        columns="Model",
        values="Rank"
    )

    # Sort by function name

    comparison_table = comparison_table.sort_index()

    # Compute summary statistics

    avg_row = comparison_table.mean(axis=0)
    std_row = comparison_table.std(axis=0)

    comparison_table.loc["Average"] = avg_row
    comparison_table.loc["Deviation"] = std_row

    # Round for display

    comparison_table = comparison_table.round(2)

    # Display results

    print(comparison_table)

    # Export to CSV

    output_name = f"rank_comparison_{METRIC}.csv"

    comparison_table.to_csv(output_name)

    print(f"\nSaved: {output_name}")



RANK COMPARISON TABLE (IGD_Mean)
Model      MGPSO  SGMGPSO  SGMGPSO_FULL  SGMGPSO_PD  SGMGPSO_TVSR  SGPSO
Function                                                                
WFG1        1.00     5.00          2.00        4.00          3.00   6.00
WFG2        2.00     4.00          5.00        6.00          1.00   3.00
WFG3        6.00     3.00          1.00        4.00          2.00   5.00
WFG4        5.00     4.00          1.00        3.00          2.00   6.00
WFG5        4.00     5.00          2.00        6.00          3.00   1.00
WFG6        6.00     5.00          1.00        3.00          4.00   2.00
WFG7        6.00     3.00          2.00        5.00          4.00   1.00
WFG8        2.00     4.00          1.00        5.00          3.00   6.00
WFG9        5.00     1.00          2.00        3.00          4.00   6.00
ZDT1        4.00     6.00          2.00        5.00          3.00   1.00
ZDT2        6.00     5.00          3.00        4.00          1.00   2.00
ZDT3        4.00 

In [2]:
import pandas as pd
import numpy as np
import os

# Configuration

METRICS = {
    "HV_Mean": False,   # maximize
    "IGD_Mean": True    # minimize
}

# Load result files (local paths)

RESULTS_DIR = "./results"

files = {
    "SGPSO": os.path.join(RESULTS_DIR, "SGPSO", "summary.csv"),
    "MGPSO": os.path.join(RESULTS_DIR, "MGPSO", "summary.csv"),
    "SGMGPSO": os.path.join(RESULTS_DIR, "SGMGPSO", "summary.csv"),
    "SGMGPSO_PD": os.path.join(RESULTS_DIR, "SGMGPSO_PP_PD", "summary.csv"),
    "SGMGPSO_TVSR": os.path.join(RESULTS_DIR, "SGMGPSO_TVSR", "summary.csv"),
    "SGMGPSO_FULL": os.path.join(RESULTS_DIR, "SGMGPSO_FULL", "summary.csv")
}

# Load datasets

dfs = []

for model_name, path in files.items():

    temp = pd.read_csv(path)

    temp.columns = temp.columns.str.strip()

    temp["Model"] = model_name

    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

# Aggregate data

df = df.groupby(["Function", "Model"], as_index=False).mean(numeric_only=True)

# Win/loss/rank table generator

def generate_win_loss_rank_table(df, metric, minimize=True):

    models = list(df["Model"].unique())
    functions = list(df["Function"].unique())

    results = []

    # Compute per-function ranks

    rank_df = df.copy()

    rank_df["Rank"] = rank_df.groupby("Function")[metric] \
        .rank(method="min", ascending=minimize)

    # Evaluate each algorithm

    for model_i in models:

        wins_per_func = []
        losses_per_func = []
        net_per_func = []
        rank_per_func = []

        for func in functions:

            sub = df[df["Function"] == func].set_index("Model")

            rank_sub = rank_df[rank_df["Function"] == func] \
                .set_index("Model")

            if model_i not in sub.index:
                continue

            val_i = sub.loc[model_i, metric]

            wins = 0
            losses = 0

            for model_j in models:

                if model_i == model_j:
                    continue

                val_j = sub.loc[model_j, metric]

                # Minimization comparison

                if minimize:

                    if val_i < val_j:
                        wins += 1

                    elif val_i > val_j:
                        losses += 1

                # Maximization comparison

                else:

                    if val_i > val_j:
                        wins += 1

                    elif val_i < val_j:
                        losses += 1

            # Net score

            net = wins - losses

            # Rank

            rank_val = rank_sub.loc[model_i, "Rank"]

            # Store results

            wins_per_func.append(wins)
            losses_per_func.append(losses)
            net_per_func.append(net)
            rank_per_func.append(rank_val)

        # Append row entries

        results.append(
            [model_i, "Wins"] +
            wins_per_func +
            [np.sum(wins_per_func)]
        )

        results.append(
            [model_i, "Losses"] +
            losses_per_func +
            [np.sum(losses_per_func)]
        )

        results.append(
            [model_i, "Net"] +
            net_per_func +
            [np.sum(net_per_func)]
        )

        results.append(
            [model_i, "Rank"] +
            rank_per_func +
            [round(np.mean(rank_per_func), 2)]
        )

    # Construct output table

    columns = ["Algorithm", "Result"] + functions + ["Sum"]

    table = pd.DataFrame(results, columns=columns)

    return table

# Generate comparison tables

for metric, minimize in METRICS.items():

    print("\n" + "="*120)
    print(f"{metric} WIN / LOSS / NET / RANK TABLE")
    print("="*120)

    table = generate_win_loss_rank_table(
        df,
        metric,
        minimize
    )

    print(table.to_string(index=False))

    # Export to CSV

    output_file = f"{metric}_win_loss_rank_table.csv"

    table.to_csv(output_file, index=False)

    print(f"\nSaved: {output_file}")



HV_Mean WIN / LOSS / NET / RANK TABLE
   Algorithm Result  WFG1  WFG2  WFG3  WFG4  WFG5  WFG6  WFG7  WFG8  WFG9  ZDT1  ZDT2  ZDT3  ZDT4  ZDT6    Sum
       MGPSO   Wins   3.0   3.0   0.0   2.0   2.0   0.0   1.0   4.0   1.0   2.0   0.0   2.0   0.0   3.0  23.00
       MGPSO Losses   2.0   2.0   5.0   3.0   3.0   5.0   4.0   1.0   4.0   3.0   5.0   3.0   0.0   2.0  42.00
       MGPSO    Net   1.0   1.0  -5.0  -1.0  -1.0  -5.0  -3.0   3.0  -3.0  -1.0  -5.0  -1.0   0.0   1.0 -19.00
       MGPSO   Rank   3.0   3.0   6.0   4.0   4.0   6.0   5.0   2.0   5.0   4.0   6.0   4.0   1.0   3.0   4.00
     SGMGPSO   Wins   1.0   2.0   2.0   3.0   1.0   1.0   3.0   2.0   4.0   0.0   1.0   1.0   0.0   5.0  26.00
     SGMGPSO Losses   4.0   3.0   3.0   2.0   4.0   4.0   2.0   3.0   1.0   5.0   4.0   4.0   0.0   0.0  39.00
     SGMGPSO    Net  -3.0  -1.0  -1.0   1.0  -3.0  -3.0   1.0  -1.0   3.0  -5.0  -3.0  -3.0   0.0   5.0 -13.00
     SGMGPSO   Rank   5.0   4.0   4.0   3.0   5.0   5.0   3.0   4.0   2.0